# 02 - Preprocessing Sparkov avec le pipeline P2




## 1. Imports

On importe les bibliothèques utiles, puis les fonctions projet pour charger les données, créer la zone `development`, séparer `X/y` et construire le pipeline.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "ml" / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

ML_SRC = PROJECT_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

from data.load_data import load_training_dataset, split_chronologically, summarize_splits
from preprocessing.pipeline import build_preprocessing_pipeline, split_features_target, get_excluded_columns, REQUIRED_COLUMNS

pd.set_option("display.max_columns", 120)
print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = C:\Users\Lenovo\Documents\fintech-mlops-lsi


## 2. Chargement de `fraudTrain.csv`

On charge uniquement le fichier de développement du projet. Si le fichier est absent, le notebook affiche le chemin attendu et s'arrête proprement.

In [2]:
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "fraudTrain.csv"
print(TRAIN_PATH)
print(TRAIN_PATH.exists())

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        "fraudTrain.csv est introuvable. Place le fichier dans data/raw/fraudTrain.csv."
    )

df, train_summary = load_training_dataset(TRAIN_PATH)
print(train_summary)
df[["trans_date_trans_time", "amt", "category", "gender", "state", "city_pop", "is_fraud"]].head()

C:\Users\Lenovo\Documents\fintech-mlops-lsi\data\raw\fraudTrain.csv
True
DatasetSummary(name='fraudTrain.csv', rows=1296675, columns=14, fraud_rate=0.005788651743883394)


,trans_date_trans_time,amt,category,gender,state,city_pop,is_fraud
0,2019-01-01 00:00:18,4.97,misc_net,F,NC,3495,0
1,2019-01-01 00:00:44,107.23,grocery_pos,F,WA,149,0
2,2019-01-01 00:00:51,220.11,entertainment,M,ID,4154,0
3,2019-01-01 00:01:16,45.00,gas_transport,M,MT,1939,0
4,2019-01-01 00:03:06,41.96,misc_pos,M,VA,99,0


## 3. Découpage chronologique

Le pipeline est ajusté uniquement sur la zone `development`. Les zones de calibration ne sont pas transformées ici.

In [3]:
splits = split_chronologically(df, sort_if_needed=True)
df_dev = splits["development"]
split_summary = summarize_splits(splits)
split_summary

,zone,start_date,end_date,rows,fraud_count,fraud_rate
0,development,2019-01-01 00:00:18,2019-12-28 17:18:24,907672,5121,0.005642
1,probability_calibration,2019-12-28 17:18:38,2020-04-03 17:53:52,194501,1252,0.006437
2,conformal_calibration,2020-04-03 17:54:44,2020-06-21 12:13:37,194502,1133,0.005825


## 4. Séparation `X` et `y`

La cible `is_fraud` est séparée des variables d'entrée avant le preprocessing.

In [4]:
X_dev, y_dev = split_features_target(df_dev)

print("X_dev :", X_dev.shape)
print("y_dev :", y_dev.shape)
print("Taux de fraude development :", f"{y_dev.mean():.6f}")

X_dev : (907672, 13)
y_dev : (907672,)
Taux de fraude development : 0.005642


### Colonnes exclues et justification

Les champs bruts requis et les champs exclus sont visibles ci-dessous. Un champ requis peut produire plusieurs sorties derivees ; leur nombre n'est pas celui des colonnes brutes retenues.

In [5]:
excluded_columns = get_excluded_columns()
print("Colonnes brutes dans X_dev :", X_dev.shape[1])
print("Champs bruts requis :", len(REQUIRED_COLUMNS), REQUIRED_COLUMNS)
exclusions_df = pd.DataFrame(
    [{"colonne": name, "justification": reason, "presente_dans_X": name in X_dev.columns}
     for name, reason in excluded_columns.items()]
)
unexplained = set(X_dev.columns) - set(REQUIRED_COLUMNS) - set(excluded_columns)
assert not unexplained, f"Colonnes sans justification : {unexplained}"
exclusions_df

Colonnes brutes dans X_dev : 13
Champs bruts requis : 11 ['trans_date_trans_time', 'dob', 'amt', 'category', 'gender', 'state', 'city_pop', 'lat', 'long', 'merch_lat', 'merch_long']


,colonne,justification,presente_dans_X
0,Unnamed: 0,colonne parasite d'index CSV,False
1,trans_num,"identifiant de transaction, reserve a l'idempo...",True
2,cc_num,"identifiant carte, hache cote customer-service",True
3,unix_time,identifiant temporel redondant avec trans_date...,False
4,first,donnee personnelle exclue,False
5,last,donnee personnelle exclue,False
6,street,donnee personnelle exclue,False
7,zip,donnee personnelle exclue,False
8,merchant,"forte cardinalite, risque de sur-apprentissage",False
9,job,"forte cardinalite, risque de sur-apprentissage",False


## 5. Construction et ajustement du pipeline

On utilise le pipeline de production P2. Il crée les variables dérivées, encode les variables catégorielles et applique la mise à l'échelle prévue. Aucun modèle de classification n'est entraîné.

In [6]:
preprocessing_pipeline = build_preprocessing_pipeline()
X_dev_processed = preprocessing_pipeline.fit_transform(X_dev, y_dev)
feature_names = preprocessing_pipeline.get_feature_names_out()

print("Matrice prétraitée :", X_dev_processed.shape)
print("Nombre de features :", len(feature_names))
print("Valeurs finies :", bool(np.isfinite(X_dev_processed).all()))

Matrice prétraitée : (907672, 76)
Nombre de features : 76
Valeurs finies : True


## 6. Noms des features et preuve d'alignement

Le CSV de reference est lu et compare aux sorties du pipeline : memes noms, meme ordre. Une divergence arrete l'execution.

In [7]:
feature_list = pd.read_csv(PROJECT_ROOT / "docs" / "feature_list_p2.csv")
documented_names = feature_list.loc[
    feature_list["section"].eq("pipeline_output"), "output_name"
].tolist()
assert feature_names.tolist() == documented_names, "Noms ou ordre des features differents du CSV."
print("Alignement noms et ordre confirme :", len(documented_names), "features")
pd.DataFrame({"feature": feature_names})

Alignement noms et ordre confirme : 76 features


,feature
0,numeric__amount_log
1,numeric__age
2,numeric__distance_km
3,numeric__city_pop_log
4,unscaled__hour_sin
...,...
71,categorical__state_VT
72,categorical__state_WA
73,categorical__state_WI
74,categorical__state_WV


## 7. Vérification des valeurs manquantes

Le pipeline retourne une matrice numérique. On vérifie qu'elle ne contient ni `NaN`, ni valeur infinie.

In [8]:
# Verifications par blocs pour limiter aussi la memoire temporaire.
missing_count = 0
finite_ok = True
for start in range(0, len(X_dev_processed), 10000):
    block = X_dev_processed[start:start + 10000]
    missing_count += int(np.isnan(block).sum())
    finite_ok = finite_ok and bool(np.isfinite(block).all())
assert missing_count == 0, "NaN dans la matrice pretraitee."
assert finite_ok, "Valeur infinie dans la matrice pretraitee."
print("Nombre total de NaN :", missing_count)
print("Toutes les valeurs sont finies :", finite_ok)

Nombre total de NaN : 0
Toutes les valeurs sont finies : True


## 8. Resume de is_night

L'indicateur 0/1 n'est plus standardise. La definition vient uniquement du pipeline de production.

In [9]:
# Lecture metier depuis le transformeur de production, par blocs.
night_parts = []
for start in range(0, len(X_dev), 10000):
    engineered_block = preprocessing_pipeline.named_steps["features"].transform(
        X_dev.iloc[start:start + 10000]
    )
    night_parts.append(engineered_block["is_night"].to_numpy())
night = np.concatenate(night_parts)
night_data = pd.DataFrame({"is_night": night, "is_fraud": y_dev.to_numpy()})
is_night_summary = night_data.groupby("is_night")["is_fraud"].agg(
    transactions="size", fraudes="sum", taux_fraude="mean"
).reset_index()
print("Heures de nuit configurees :", preprocessing_pipeline.named_steps["features"].night_hours)
print("Colonne pipeline disponible :", "unscaled__is_night" in feature_names)
is_night_summary

Heures de nuit configurees : (22, 23, 0, 1, 2, 3)
Colonne pipeline disponible : True


,is_night,transactions,fraudes,taux_fraude
0,0,694662,787,0.001133
1,1,213010,4334,0.020346


## 9. Aperçu de la matrice prétraitée

On affiche quelques lignes pour contrôler la forme finale du livrable de preprocessing.

In [10]:
# Seulement cinq lignes sont enveloppees dans un DataFrame.
pd.DataFrame(X_dev_processed[:5], columns=feature_names)

,numeric__amount_log,numeric__age,numeric__distance_km,numeric__city_pop_log,unscaled__hour_sin,unscaled__hour_cos,unscaled__day_of_week_sin,unscaled__day_of_week_cos,unscaled__is_weekend,unscaled__is_night,gender__gender_M,categorical__category_entertainment,categorical__category_food_dining,categorical__category_gas_transport,categorical__category_grocery_net,categorical__category_grocery_pos,categorical__category_health_fitness,categorical__category_home,categorical__category_kids_pets,categorical__category_misc_net,categorical__category_misc_pos,categorical__category_personal_care,categorical__category_shopping_net,categorical__category_shopping_pos,categorical__category_travel,categorical__state_AK,categorical__state_AL,categorical__state_AR,categorical__state_AZ,categorical__state_CA,categorical__state_CO,categorical__state_CT,categorical__state_DC,categorical__state_DE,categorical__state_FL,categorical__state_GA,categorical__state_HI,categorical__state_IA,categorical__state_ID,categorical__state_IL,categorical__state_IN,categorical__state_KS,categorical__state_KY,categorical__state_LA,categorical__state_MA,categorical__state_MD,categorical__state_ME,categorical__state_MI,categorical__state_MN,categorical__state_MO,categorical__state_MS,categorical__state_MT,categorical__state_NC,categorical__state_ND,categorical__state_NE,categorical__state_NH,categorical__state_NJ,categorical__state_NM,categorical__state_NV,categorical__state_NY,categorical__state_OH,categorical__state_OK,categorical__state_OR,categorical__state_PA,categorical__state_RI,categorical__state_SC,categorical__state_SD,categorical__state_TN,categorical__state_TX,categorical__state_UT,categorical__state_VA,categorical__state_VT,categorical__state_WA,categorical__state_WI,categorical__state_WV,categorical__state_WY
0,-1.353749,-0.883114,0.084742,-0.084478,0.0,1.0,0.781832,0.62349,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.893814,-0.307542,-1.577491,-1.368406,0.0,1.0,0.781832,0.62349,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,1.447966,0.613372,1.101914,-0.014062,0.0,1.0,0.781832,0.62349,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.230123,0.325586,0.671360,-0.324621,0.0,1.0,0.781832,0.62349,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.177087,-0.767999,0.048986,-1.533738,0.0,1.0,0.781832,0.62349,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
